# Data Quality Monitoring System for E-Commerce Operations

## Project Overview

This project presents the development of a Data Quality Monitoring System using the **Olist Brazilian E-Commerce Public Dataset**. The objective is to simulate a real-world business analytics workflow by identifying, assessing, and improving the quality of transactional data before it is used for reporting and decision-making.

The project follows an end-to-end data analytics pipeline, beginning with data preparation in **Python**, followed by data modelling and quality analysis in **PostgreSQL**, and concluding with interactive **Power BI** dashboards that monitor key data quality metrics and business performance indicators.

Throughout the project, data quality dimensions such as **completeness, accuracy, consistency, validity, uniqueness, and timeliness** are evaluated. The datasets are cleaned, standardized, validated, and transformed into analysis-ready data to support reliable business intelligence and operational reporting.

This project demonstrates practical skills in data cleaning, exploratory data analysis, SQL development, data quality assessment, and dashboard development while following industry-standard data analytics practices.

This script focuses on the preparation of the Olist products dataset as part of the Data Quality Monitoring System for E-Commerce Operations. The cleaned dataset supports product and inventory analysis by providing product category information, physical attributes, and logistics-related measurements linked to products sold through the marketplace. Preparing this dataset ensures that product information is complete, consistent, and reliable for downstream analysis in PostgreSQL, SQL, and Power BI, enabling accurate product classification, inventory reporting, logistics analysis, and sales performance evaluation.


### Import the Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 
import psycopg2
from sqlalchemy import create_engine 
from pathlib import Path

### Load The dataset

In [3]:
products_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_products_dataset.csv")

## **ORDER PRODUCTS DATASET**

### 1. Data Inspection

In [4]:
# The first 5 rows of the products dataset
products_df.head(5)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [5]:
# The number of rows and columns in the products dataset
products_df.shape

(32951, 9)

In [6]:
# The columns and their data types in the products dataset
products_df.dtypes

product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

The identifier columns **`product_id`** and **`product_category_name`** have been correctly recognised as text (`object`), while the remaining product attribute columns have been imported as numeric (`float64`) values.

The use of the `float64` data type for the numeric attributes is expected at this stage, as Pandas automatically converts integer-based columns containing missing values to floating-point numbers. No data type inconsistencies requiring immediate correction were identified during the initial inspection. The current data types are therefore suitable.

In [7]:
# The number of missing values in each column of the products dataset
products_df.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

The missing values are concentrated in specific product attributes rather than being distributed randomly. The descriptive columns **`product_category_name`**, **`product_name_lenght`**, **`product_description_lenght`**, and **`product_photos_qty`** each contain **610** missing values, while the physical measurement columns **`product_weight_g`**, **`product_length_cm`**, **`product_height_cm`**, and **`product_width_cm`** each contain **2** missing values.

The consistent distribution of missing values suggests that these records may be related and should be investigated further during the data profiling stage. 

In [8]:
# The number of duplicate rows in the products dataset
for col in products_df.columns:
    duplicate_rows = products_df[col].duplicated().sum()
    print(f"{col}: {duplicate_rows}")

product_id: 0
product_category_name: 32877
product_name_lenght: 32884
product_description_lenght: 29990
product_photos_qty: 32931
product_weight_g: 30746
product_length_cm: 32851
product_height_cm: 32848
product_width_cm: 32855


The duplicate value inspection confirms that **`product_id`** contains no duplicate values, indicating that each product is uniquely identified within the dataset. This is consistent with the expected behaviour of a primary identifier and supports the integrity of relationships between the Products dataset and other tables within the integrated e-commerce database.

The remaining columns contain a substantial number of duplicate values; however, these duplicates are expected because they represent shared product characteristics rather than repeated records. Product categories, dimensions, weights, description lengths, and photo quantities can legitimately occur across multiple products. Consequently, these results do not indicate a data quality issue.

In [9]:
# statistical summary of the products dataset numerical columns
products_df.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [10]:
# statistical summary of the products dataset categorical columns
products_df.describe(include=[object])

,product_id,product_category_name
count,32951,32341
unique,32951,73
top,1e9e8ef04dbcff4541ed26657ea517e5,cama_mesa_banho
freq,1,3029


The statistical summary indicates that the Products dataset exhibits realistic numerical distributions and a consistent categorical structure. The identifier **`product_id`** remains unique across all records, while **`product_category_name`** contains **73** distinct product categories, reflecting a diverse product catalogue suitable for downstream analysis.

The numerical attributes fall within generally reasonable ranges for product characteristics. However, the minimum value of **0 grams** recorded in **`product_weight_g`** has been identified as a potential anomaly requiring further investigation during the data profiling stage. At this stage of the inspection, no evidence suggests that the remaining numerical distributions or categorical values compromise the overall structural quality of the dataset.

### 2. Data Profiling

#### Missing Values

In [11]:
# Percentage of missing values in the products dataset
round((products_df.isnull().sum() / len(products_df)) * 100, 2)

product_id                    0.00
product_category_name         1.85
product_name_lenght           1.85
product_description_lenght    1.85
product_photos_qty            1.85
product_weight_g              0.01
product_length_cm             0.01
product_height_cm             0.01
product_width_cm              0.01
dtype: float64

In [12]:
# Display records containing missing values
products_df[products_df.isnull().any(axis=1)]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [13]:
# Check whether the same records contain the missing values
products_df[products_df.isnull().any(axis=1)].isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

The missing value investigation indicates that the Products dataset is highly complete, with missing values affecting only a small proportion of the total records. Two distinct missing value patterns were identified during profiling. The descriptive attributes **`product_category_name`**, **`product_name_lenght`**, **`product_description_lenght`**, and **`product_photos_qty`** each contain **610** missing values (**1.85%**), while the physical measurement attributes **`product_weight_g`**, **`product_length_cm`**, **`product_height_cm`**, and **`product_width_cm`** each contain only **2** missing values (**0.01%**).

Further investigation confirmed that the missing values are systematic rather than randomly distributed. The **610** affected records consistently lack descriptive product information while retaining valid physical measurements, whereas the remaining **2** records contain the opposite pattern. This suggests that the missing values originate from two separate data quality scenarios rather than isolated data entry errors.

Overall, the dataset demonstrates a high level of completeness, with more than **98%** of records containing complete information. The identified missing value patterns will be addressed during the data cleaning phase using an approach that preserves data integrity.

#### Duplicates

In [14]:
# Number of duplicate rows in the products dataset
print(products_df.duplicated().sum())

0


In [15]:
# Display duplicate rows
products_df[products_df.duplicated(keep=False)].sort_values(by="product_id")

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


The duplicate row investigation confirmed that the Products dataset contains **no duplicate records**, indicating that each product is represented by a single, unique observation. Verification using the duplicate row output further confirmed this result, as no duplicated records were returned.

Since no duplicate rows were identified, no cleaning is required for this aspect of the dataset, and the uniqueness of the product records can be retained during the data loading process.

#### Product Identifier Validation

In [16]:
# Check for missing product IDs
print(products_df["product_id"].isnull().sum())

0


In [17]:
# Check for duplicate product IDs
print(products_df["product_id"].duplicated().sum())

0


In [18]:
# Validate the length of product IDs
products_df["product_id"].str.len().value_counts().sort_index()

product_id
32    32951
Name: count, dtype: int64

The product identifier validation confirms that **`product_id`** demonstrates a high level of data integrity across the Products dataset. No missing or duplicate product identifiers were identified, indicating that each product is uniquely represented.

Additionally, all **32,951** product identifiers were found to have a consistent length of **32 characters**, confirming a standardised identifier format throughout the dataset. This consistency reduces the risk of key mismatches during database integration and supports accurate relational joins. Based on these findings, **`product_id`** satisfies the data quality requirements.

#### Text Quality Assessment

In [19]:
# Text quality assessment for product category names

# Leading or trailing spaces
leading_trailing_spaces = products_df[
    products_df["product_category_name"].notna() &
    (
        products_df["product_category_name"] !=
        products_df["product_category_name"].str.strip()
    )
]

# Multiple consecutive spaces
multiple_spaces = products_df[
    products_df["product_category_name"].fillna("").str.contains(r"\s{2,}", regex=True)
]

# Unexpected special characters
special_characters = products_df[
    products_df["product_category_name"].fillna("").str.contains(r"[^a-zA-Z0-9_ ]", regex=True)
]

# Character encoding issues
encoding_issues = products_df[
    products_df["product_category_name"].fillna("").str.contains(r"�|Ã|Â|â", regex=True)
]

# Case consistency
case_consistency = products_df["product_category_name"].dropna().str.islower().value_counts()

# Display results
print("TEXT QUALITY ASSESSMENT")


print(f"\nLeading/Trailing Spaces: {len(leading_trailing_spaces)} record(s)")
print(f"Multiple Consecutive Spaces: {len(multiple_spaces)} record(s)")
print(f"Unexpected Special Characters: {len(special_characters)} record(s)")
print(f"Character Encoding Issues: {len(encoding_issues)} record(s)")

print("\nCase Consistency:")
print(case_consistency)

# Display records only when issues are found
if len(leading_trailing_spaces) > 0:
    print("\nRecords with Leading/Trailing Spaces")
    display(leading_trailing_spaces)

if len(multiple_spaces) > 0:
    print("\nRecords with Multiple Consecutive Spaces")
    display(multiple_spaces)

if len(special_characters) > 0:
    print("\nRecords with Unexpected Special Characters")
    display(special_characters)

if len(encoding_issues) > 0:
    print("\nRecords with Character Encoding Issues")
    display(encoding_issues)

TEXT QUALITY ASSESSMENT

Leading/Trailing Spaces: 0 record(s)
Multiple Consecutive Spaces: 0 record(s)
Unexpected Special Characters: 0 record(s)
Character Encoding Issues: 0 record(s)

Case Consistency:
product_category_name
True    32341
Name: count, dtype: int64


The text quality assessment confirmed that **`product_category_name`** is well standardised and exhibits a high level of textual consistency. No leading or trailing whitespace, multiple consecutive spaces, unexpected special characters, or character encoding issues were identified during profiling. Additionally, all non-null category names were found to be stored consistently in lowercase, indicating a uniform text formatting standard across the dataset.


#### Numeric Validation

In [20]:
# Count zero values in numeric columns
numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

(products_df[numeric_columns] == 0).sum()

product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              4
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

In [21]:
# Count negative values in numeric columns
(products_df[numeric_columns] < 0).sum()

product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

In [22]:
# Display products containing zero values
products_df[
    (products_df[numeric_columns] == 0).any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


The numeric value validation indicates that the Products dataset maintains a high level of numerical integrity. No negative values were identified across the assessed numeric attributes, confirming that the recorded product measurements and quantities remain within logically valid ranges.

The only anomaly identified during profiling was **4** records in **`product_weight_g`** with a recorded weight of **0 grams**. Further investigation showed that these records contain complete descriptive information and valid product dimensions, indicating that the zero values are isolated to the product weight attribute. Although these values may represent data entry errors or placeholder measurements, the available evidence is insufficient to conclude that they are invalid.

To preserve the integrity of the original data, the zero-weight records will be retained during the cleaning process and documented as potential anomalies rather than being modified or removed without business justification.The numeric value validation indicates that the Products dataset maintains a high level of numerical integrity. No negative values were identified across the assessed numeric attributes, confirming that the recorded product measurements and quantities remain within logically valid ranges.

The only anomaly identified during profiling was **4** records in **`product_weight_g`** with a recorded weight of **0 grams**. Further investigation showed that these records contain complete descriptive information and valid product dimensions, indicating that the zero values are isolated to the product weight attribute. Although these values may represent data entry errors or placeholder measurements, the available evidence is insufficient to conclude that they are invalid.

To preserve the integrity of the original data, the zero-weight records will be retained during the cleaning process and documented as potential anomalies rather than being modified or removed without business justification.

#### Extreme Value Assessment

In [23]:
# Statistical summary of numeric columns
products_df[numeric_columns].describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [24]:
# Display products with unusually high weights
products_df.sort_values(by="product_weight_g", ascending=False).head(10)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
25166,26644690fde745fc4654719c3904e1db,cama_mesa_banho,59.0,534.0,1.0,40425.0,13.0,65.0,28.0
2417,343c15a347e523f2b6cf38a5db81e179,esporte_lazer,48.0,1232.0,2.0,30000.0,105.0,65.0,20.0
4814,0984eaa8480e41aded35bd7b5131a1c1,beleza_saude,58.0,540.0,1.0,30000.0,55.0,75.0,61.0
15982,c100e5fef1abb5e1c5054d1dac2d83ac,beleza_saude,57.0,534.0,1.0,30000.0,55.0,75.0,61.0
23581,a8baceb529f7e2a5c770cc5b4e3da35d,beleza_saude,55.0,432.0,1.0,30000.0,55.0,75.0,61.0
7234,8d6f2c3454002d3f5aa7479a7fad7794,moveis_sala,52.0,1105.0,1.0,30000.0,80.0,60.0,60.0
31113,ed6e17c5f34a20c7569b6ac2597ee223,automotivo,53.0,611.0,1.0,30000.0,105.0,20.0,50.0
26724,cd1db0c97e4b3644bb52432611e09c58,moveis_decoracao,52.0,583.0,2.0,30000.0,88.0,45.0,41.0
4594,e6574cebbeb21c82802d3a0b682129b6,esporte_lazer,49.0,1263.0,2.0,30000.0,105.0,65.0,25.0
4655,027293c3b6d9e221268d9d6a5ffe5d0b,industria_comercio_e_negocios,57.0,148.0,1.0,30000.0,80.0,50.0,50.0


In [25]:
# Display products with unusually large dimensions
products_df.sort_values(by="product_length_cm", ascending=False).head(10)

products_df.sort_values(by="product_height_cm", ascending=False).head(10)

products_df.sort_values(by="product_width_cm", ascending=False).head(10)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
26970,b17808303e15dd50538c011b44295427,cama_mesa_banho,56.0,502.0,3.0,1050.0,23.0,93.0,118.0
19326,6d30e5e702df2b8719d9c6be1bdf425b,cool_stuff,44.0,909.0,4.0,25000.0,27.0,23.0,105.0
475,a2f4e28e50f60566eeb99f842ffc0fd9,instrumentos_musicais,57.0,374.0,1.0,5850.0,40.0,9.0,105.0
28739,3758055ab2434bd36ac78e00b15b5cf6,beleza_saude,52.0,1193.0,3.0,600.0,50.0,8.0,105.0
23725,cb428376d66b5e216d2ef9f3b27fc172,instrumentos_musicais,59.0,209.0,2.0,5000.0,45.0,14.0,105.0
25181,3b17f6528c9e2a01b2f75f844a60ddae,instrumentos_musicais,59.0,326.0,1.0,5000.0,40.0,9.0,105.0
31050,83d68ad4e5707409089afff26a40b2df,brinquedos,33.0,112.0,1.0,13150.0,50.0,14.0,104.0
15308,e7248872169a7ab67e20c182aaf17976,instrumentos_musicais,60.0,327.0,1.0,5000.0,35.0,9.0,103.0
31163,e54c0428a8cf1b79f63823b92e20aacc,moveis_cozinha_area_de_servico_jantar_e_jardim,42.0,440.0,1.0,25800.0,39.0,13.0,102.0
10256,68e6e8fd8c5f5b252b105d00daa9b57b,moveis_escritorio,51.0,758.0,1.0,13800.0,39.0,13.0,102.0


The dataset contains a small number of products with comparatively large weights and dimensions; however, further investigation found no evidence that these values are erroneous or inconsistent with the nature of the products being represented. The reviewed records span multiple product categories, suggesting that the observed extremes reflect genuine variation within the e-commerce catalogue rather than systematic data entry errors.

The statistical distributions of the numeric attributes remain reasonable, and no implausible measurements are identified.

#### Cross-Field Consistency

In [26]:
# Check whether the same records contain missing descriptive fields
products_df[
    products_df[
        [
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty"
        ]
    ].isnull().any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [27]:
# Check products with missing physical measurements
products_df[
    products_df[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isnull().any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The missing values within the dataset follow structured and consistent patterns rather than occurring randomly across individual attributes. Products with missing descriptive information consistently lack **`product_category_name`**, **`product_name_lenght`**, **`product_description_lenght`**, and **`product_photos_qty`**, while retaining complete physical measurement information. This indicates that the descriptive metadata is missing as a group rather than through isolated data entry errors.

Further investigation identified **2** records with missing physical measurements. One record retains complete descriptive information but lacks all physical measurement attributes, whereas the second record is missing both descriptive and physical measurement information. This record represents the highest level of incompleteness identified during profiling and will require careful consideration during the data cleaning stage.

#### 3. Referential Intergrity Check for Missing values

Before deciding whether to remove records with missing values, a referential integrity check was performed to determine whether the affected products are referenced within the **Order Items** dataset. Since the e-commerce datasets form an integrated relational database rather than standalone tables, removing product records without verifying their relationships could compromise the integrity of downstream analyses. This assessment ensures that any cleaning decisions preserve valid relationships between products and customer transactions.

In [28]:
order_items_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\cleaned\olist_order_items_cleaned.csv")

In [29]:
# Extract products with missing values

products_with_missing = products_df[
    products_df.isnull().any(axis=1)
]

products_with_missing[["product_id"]].head()

,product_id
105,a41e356c76fab66334f36de622ecbd3a
128,d8dee61c2034d6d075997acef1870e9b
145,56139431d72cd51f19eb9f7dae4d1617
154,46b48281eb6d663ced748f324108c733
197,5fb61f482620cb672f5e586bb132eae9


In [30]:
# Find products with missing values that appear in the Order Items dataset

products_in_orders = products_with_missing.merge(
    order_items_df[["product_id"]],
    on="product_id",
    how="inner"
)

print(f"Products with missing values found in Order Items: {len(products_in_orders)}")
products_in_orders.head()

Products with missing values found in Order Items: 1604


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
1,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
2,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
3,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
4,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0


In [31]:
# Find products with missing values that never appear in the Order Items dataset

products_not_in_orders = products_with_missing[
    ~products_with_missing["product_id"].isin(order_items_df["product_id"])
]

print(f"Products with missing values NOT found in Order Items: {len(products_not_in_orders)}")
products_not_in_orders.head()

Products with missing values NOT found in Order Items: 0


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


In [32]:
# Summary of referential integrity assessment

total_missing_products = products_with_missing["product_id"].nunique()
referenced_products = products_in_orders["product_id"].nunique()
not_referenced_products = products_not_in_orders["product_id"].nunique()

print("Referential Integrity Assessment:")
print(f"Total products with missing values: {total_missing_products}")
print(f"Unique products referenced in Order Items: {referenced_products}")
print(f"Unique products not referenced in Order Items: {not_referenced_products}")

Referential Integrity Assessment:
Total products with missing values: 611
Unique products referenced in Order Items: 611
Unique products not referenced in Order Items: 0


All **609** products containing missing values are referenced within the **Order Items** dataset, collectively appearing in **1,586** customer order records. This demonstrates that every incomplete product record participates in legitimate business transactions and forms part of the integrated e-commerce database.

Removing these records would compromise referential integrity by creating unmatched product references during database joins and analytical reporting. Based on these findings, no records will be removed from the Products dataset. 

### 3. Data Cleaning

#### **Duplicate Records**

The duplicate row investigation confirmed that the Products dataset contains no duplicate records. Since no duplicate observations were identified, no cleaning was required. The original records were retained without modification to preserve the integrity of the dataset.

#### **Product Identifier Integrity**

The **`product_id`** field was found to contain no missing values, no duplicate values, and a consistent identifier length across all records. As the identifier satisfied the required data quality standards for completeness, uniqueness, and consistency, no corrective action was necessary.

#### **Text Quality**

The text quality assessment found no leading or trailing whitespace, multiple consecutive spaces, unexpected special characters, inconsistent casing, or character encoding issues within **`product_category_name`**. Consequently, no text standardisation or cleaning was performed.

#### **Zero Product Weights**

Four records were identified with a recorded product weight of **0 grams**. Although these values may represent anomalies, no supporting evidence confirmed that they were incorrect. To preserve the integrity of the original source data and avoid introducing unsupported assumptions, the values were retained and documented rather than modified.

#### **Extreme Values**

Products with comparatively large weights and dimensions were investigated during profiling. The reviewed values were consistent with the characteristics of the associated product categories and were therefore considered legitimate observations rather than data quality errors. No outlier treatment was applied.

In [33]:
# Display the final dataset dimensions
print(f"Final dataset shape: {products_df.shape}")

# Check for missing values
missing_values = products_df.isnull().sum()
print(missing_values)

# Verify duplicate records
print(
    f"Remaining duplicate records: "
    f"{products_df.duplicated().sum()}"
)

# Verify duplicate product IDs
duplicate_product_ids = products_df["product_id"].duplicated().sum()

print(
    f"Duplicate product IDs: "
    f"{duplicate_product_ids}"
)

# Verify product ID completeness
missing_product_ids = products_df["product_id"].isnull().sum()

print(
    f"Missing product IDs: "
    f"{missing_product_ids}"
)

# Verify text quality
remaining_whitespace_categories = products_df[
    products_df["product_category_name"]
    .fillna("")
    .str.strip()
    .eq("")
    &
    products_df["product_category_name"].notna()
]

print(
    f"Whitespace-only product categories: "
    f"{len(remaining_whitespace_categories)}"
)

# Verify negative values in numeric columns
numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

negative_values = (products_df[numeric_columns] < 0).sum().sum()

print(
    f"Negative numeric values: "
    f"{negative_values}"
)

# Display dataset information
products_df.info()

Final dataset shape: (32951, 9)
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Remaining duplicate records: 0
Duplicate product IDs: 0
Missing product IDs: 0
Whitespace-only product categories: 0
Negative numeric values: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float6

### 4. Export The Cleaned Dataset

In [34]:
PROJECT_ROOT = Path(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce")

RAW_DATA_PATH = PROJECT_ROOT / "01_datasets" / "raw"
CLEANED_DATA_PATH = PROJECT_ROOT / "01_datasets" / "cleaned"

In [35]:
products_df.to_csv(
    CLEANED_DATA_PATH / "olist_products_cleaned.csv",
    index=False
)